# Optimization, Regularization, and Debugging

Companion notebook for Sections 7 and 9 of the MLP lecture notes.

Objectives:

- compare learning-rate behavior;
- compare SGD-style training with Adam-style training;
- visualize initialization effects on activation scale;
- demonstrate overfitting and mitigation with early stopping and L2 regularization;
- build a practical debugging checklist for MLP training.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Dataset

We use `load_digits` with a deliberately small training split to make overfitting easier to observe.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings

X, y = load_digits(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, train_size=250, stratify=y_train_full, random_state=42
)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)


## 2. Learning-rate comparison

Too small can be slow. Too large can be unstable. We compare loss curves using scikit-learn's `MLPClassifier` with SGD.


In [ ]:
def fit_mlp(learning_rate_init, solver='sgd', alpha=0.0001, early_stopping=False, hidden_layer_sizes=(64,), max_iter=80):
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation='relu',
            solver=solver,
            learning_rate_init=learning_rate_init,
            alpha=alpha,
            early_stopping=early_stopping,
            validation_fraction=0.2,
            max_iter=max_iter,
            random_state=42,
        ),
    )
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        model.fit(X_train, y_train)
    return model

lr_values = [1e-5, 1e-3, 1e-1]
lr_models = {lr: fit_mlp(lr, solver='sgd', max_iter=80) for lr in lr_values}

rows = []
for lr, model in lr_models.items():
    clf = model.named_steps['mlpclassifier']
    rows.append({
        'learning_rate_init': lr,
        'epochs_run': clf.n_iter_,
        'final_loss': clf.loss_,
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'val_accuracy': accuracy_score(y_val, model.predict(X_val)),
    })

pd.DataFrame(rows)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for lr, model in lr_models.items():
    loss_curve = model.named_steps['mlpclassifier'].loss_curve_
    ax.plot(loss_curve, label=f'lr={lr}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training loss')
ax.set_title('Learning rate affects convergence')
ax.legend()
plt.show()


## 3. Optimizer comparison

Adam is usually a strong default for initial experiments. SGD can work well, but often needs more tuning.


In [ ]:
optimizer_settings = [
    {'name': 'SGD lr=1e-3', 'solver': 'sgd', 'lr': 1e-3},
    {'name': 'SGD lr=1e-2', 'solver': 'sgd', 'lr': 1e-2},
    {'name': 'Adam lr=1e-3', 'solver': 'adam', 'lr': 1e-3},
]

opt_rows = []
opt_models = {}
for setting in optimizer_settings:
    model = fit_mlp(setting['lr'], solver=setting['solver'], max_iter=100)
    opt_models[setting['name']] = model
    opt_rows.append({
        'optimizer': setting['name'],
        'epochs_run': model.named_steps['mlpclassifier'].n_iter_,
        'final_loss': model.named_steps['mlpclassifier'].loss_,
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'val_accuracy': accuracy_score(y_val, model.predict(X_val)),
    })

pd.DataFrame(opt_rows).sort_values('val_accuracy', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, model in opt_models.items():
    ax.plot(model.named_steps['mlpclassifier'].loss_curve_, label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training loss')
ax.set_title('SGD versus Adam loss curves')
ax.legend()
plt.show()


## 4. Initialization and activation scale

Good initialization keeps activations from exploding or vanishing as they pass through layers. This NumPy simulation compares naive large weights, Xavier-style scale, and He-style scale.


In [ ]:
rng = np.random.default_rng(42)

def relu(z):
    return np.maximum(0, z)

def simulate_activation_variance(width=128, depth=20, init='he'):
    A = rng.normal(size=(1000, width))
    variances = [A.var()]
    for _ in range(depth):
        if init == 'large':
            W = rng.normal(scale=2.0, size=(width, width))
        elif init == 'xavier':
            W = rng.normal(scale=np.sqrt(1 / width), size=(width, width))
        elif init == 'he':
            W = rng.normal(scale=np.sqrt(2 / width), size=(width, width))
        else:
            raise ValueError(init)
        A = relu(A @ W)
        variances.append(A.var())
    return np.array(variances)

init_curves = {name: simulate_activation_variance(init=name) for name in ['large', 'xavier', 'he']}

fig, ax = plt.subplots(figsize=(8, 5))
for name, curve in init_curves.items():
    ax.semilogy(curve, marker='o', label=name)
ax.set_xlabel('Layer')
ax.set_ylabel('Activation variance')
ax.set_title('Initialization affects signal scale through depth')
ax.legend()
plt.show()


## 5. Overfitting and regularization

A large network can memorize a small training set. L2 regularization and early stopping can reduce variance.


In [ ]:
regularization_settings = [
    {'name': 'large, weak L2, no early stopping', 'alpha': 1e-8, 'early_stopping': False},
    {'name': 'large, stronger L2', 'alpha': 1e-2, 'early_stopping': False},
    {'name': 'large, early stopping', 'alpha': 1e-4, 'early_stopping': True},
]

reg_rows = []
reg_models = {}
for setting in regularization_settings:
    model = fit_mlp(
        learning_rate_init=1e-3,
        solver='adam',
        alpha=setting['alpha'],
        early_stopping=setting['early_stopping'],
        hidden_layer_sizes=(256, 128),
        max_iter=200,
    )
    reg_models[setting['name']] = model
    reg_rows.append({
        'setting': setting['name'],
        'epochs_run': model.named_steps['mlpclassifier'].n_iter_,
        'train_accuracy': accuracy_score(y_train, model.predict(X_train)),
        'val_accuracy': accuracy_score(y_val, model.predict(X_val)),
        'test_accuracy': accuracy_score(y_test, model.predict(X_test)),
    })

pd.DataFrame(reg_rows).sort_values('val_accuracy', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, model in reg_models.items():
    clf = model.named_steps['mlpclassifier']
    ax.plot(clf.loss_curve_, label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training loss')
ax.set_title('Regularization settings change training dynamics')
ax.legend(fontsize=8)
plt.show()


## 6. Can the model overfit a tiny subset?

A common sanity check is to verify that the model can memorize a very small training set. If it cannot, the labels, loss, architecture, or optimizer setup may be wrong.


In [ ]:
tiny_X = X_train[:10]
tiny_y = y_train[:10]

tiny_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 64), solver='adam', learning_rate_init=0.01,
                  alpha=0.0, max_iter=1000, random_state=42),
)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    tiny_model.fit(tiny_X, tiny_y)

print('Tiny-subset training accuracy:', accuracy_score(tiny_y, tiny_model.predict(tiny_X)))


## 7. Debugging checklist as executable checks

Before tuning architecture, verify basic data and training assumptions.


In [ ]:
def mlp_debugging_checks(X_train, y_train, X_val, y_val, model):
    checks = []
    checks.append({'check': 'no NaNs in X_train', 'passed': not np.isnan(X_train).any()})
    checks.append({'check': 'no NaNs in X_val', 'passed': not np.isnan(X_val).any()})
    checks.append({'check': 'labels are integer encoded', 'passed': np.issubdtype(np.asarray(y_train).dtype, np.integer)})
    checks.append({'check': 'at least two classes in train', 'passed': len(np.unique(y_train)) >= 2})
    checks.append({'check': 'train and validation have same number of features', 'passed': X_train.shape[1] == X_val.shape[1]})
    checks.append({'check': 'model has loss curve', 'passed': hasattr(model.named_steps['mlpclassifier'], 'loss_curve_')})
    checks.append({'check': 'final loss finite', 'passed': np.isfinite(model.named_steps['mlpclassifier'].loss_)})
    return pd.DataFrame(checks)

mlp_debugging_checks(X_train, y_train, X_val, y_val, opt_models['Adam lr=1e-3'])


## 8. Symptom-to-fix map

This table mirrors the debugging guidance from the lecture notes.


In [ ]:
pd.DataFrame([
    {'symptom': 'loss does not decrease', 'likely cause': 'learning rate too small, bad initialization, wrong loss', 'first fix': 'increase LR slightly; check loss/output pair'},
    {'symptom': 'loss diverges or NaN', 'likely cause': 'learning rate too large, no scaling, extreme inputs', 'first fix': 'reduce LR; standardize features'},
    {'symptom': 'high train and validation loss', 'likely cause': 'underfitting', 'first fix': 'increase width/depth or train longer'},
    {'symptom': 'low train loss, high validation loss', 'likely cause': 'overfitting', 'first fix': 'early stopping, L2, dropout, more data'},
    {'symptom': 'all predictions same class', 'likely cause': 'dead units, bad labels, severe imbalance', 'first fix': 'check labels and class distribution; adjust LR/init'},
    {'symptom': 'validation much better than test', 'likely cause': 'test overuse or distribution shift', 'first fix': 'audit splits; use test once'},
])


## Exercises

1. Increase the small training set from 250 to 1000 examples. Does overfitting shrink?
2. Try `alpha=1.0`. Does strong L2 underfit?
3. Change the initialization simulation from ReLU to tanh. Which initialization behaves better?


## Takeaways

- Learning rate controls whether optimization is slow, stable, or divergent.
- Adam is a strong default, but still needs scaling and reasonable learning rates.
- Initialization affects signal propagation through deep networks.
- Regularization reduces overfitting but can cause underfitting if too strong.
- Debugging starts with basic data, labels, loss, scaling, and overfit-on-small-subset checks.
